# Notebook 1: Active memory basics

This notebook shows the simplest local workflow for agent memory: start a session, add working memory, store entity/task cards, extract claims, and retrieve the composed context.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


In [ ]:
from src.api.memory_manager import MemoryManager

memory_root = REPO_ROOT / "notebooks" / ".demo-memory-01"
shutil.rmtree(memory_root, ignore_errors=True)
manager = MemoryManager(memory_root)
session = manager.start_session(
    current_goal="Track a payment incident",
    metadata={"owner": "notebook-demo"},
)
show(session.model_dump(mode="json"))

In [ ]:
working_item = manager.add_working_item(
    item_type="observation",
    content="The Acme Payments API returned 401 responses after the certificate rotation completed.",
)
entity = manager.add_entity(
    name="Acme Payments",
    entity_type="service",
    description="Payment API used by the checkout flow.",
    attributes={"language": "python", "tier": "critical"},
)
task = manager.add_task(
    title="Investigate checkout authentication failures",
    description="Confirm whether the certificate rotation invalidated downstream tokens.",
    priority=2,
)
claims = manager.extract_claims(working_item.content, source_ref=working_item.item_id)
show({
    "working_item": working_item.model_dump(mode="json"),
    "entity": entity.model_dump(mode="json"),
    "task": task.model_dump(mode="json"),
    "claims": [claim.model_dump(mode="json") for claim in claims],
})

In [ ]:
context = manager.retrieve_context()
show({
    "session_status": context["session"]["status"],
    "working_items": len(context["working_items"]),
    "entities": len(context["entities"]),
    "tasks": len(context["tasks"]),
    "claims": len(context["claims"]),
})
show(context)

In [ ]:
created_files = sorted(
    str(path.relative_to(memory_root))
    for path in memory_root.rglob("*")
    if path.is_file()
)
show(created_files)

## What this notebook demonstrates

- filesystem-backed active memory
- session state and working memory updates
- entity/task cards as structured memory
- claim extraction feeding retrieval context